# Notebook 04 — Fee Revenue Estimation

**Goal:** Estimate how much Kalshi has earned in fees from exotic parlays since launch.

**Approach:**
1. Start with **taker fees only** (the bigger, more visible side)
2. Apply Kalshi's published formula: `taker_fee = 0.07 × contracts × P × (1-P)`
3. Sanity check total against industry benchmark (~1% of notional)
4. Break down by series, month, and price bucket
5. Layer in maker fees (`0.0175 × contracts × P × (1-P)`)

**Why this matters:** Kalshi doesn't publish fee revenue by product. This lets us estimate exotic parlay revenue with reasonable confidence.

**Data source:** `kalshi.trade_report` via Dune Analytics — pulled fresh, not cached.

In [ ]:
import requests
import time
import pandas as pd

API_KEY = "YOUR_DUNE_API_KEY"  # paste your key
headers = {"X-Dune-API-Key": API_KEY}

def run_fresh(sql, name="Notebook 04 query"):
    """Create + execute a Dune query and return results as a DataFrame.
    Pulls fresh data — does not use cached results."""
    r = requests.post(
        "https://api.dune.com/api/v1/query",
        headers={**headers, "Content-Type": "application/json"},
        json={"name": name, "query_sql": sql, "is_private": True}
    )
    qid = r.json()["query_id"]
    
    r = requests.post(f"https://api.dune.com/api/v1/query/{qid}/execute", headers=headers)
    eid = r.json()["execution_id"]
    
    while True:
        time.sleep(4)
        state = requests.get(f"https://api.dune.com/api/v1/execution/{eid}/status", headers=headers).json()["state"]
        print(f"  {state}")
        if state == "QUERY_STATE_COMPLETED":
            break
    
    rows = requests.get(f"https://api.dune.com/api/v1/execution/{eid}/results", headers=headers).json()["result"]["rows"]
    return pd.DataFrame(rows), qid

## Step 1 — Total Taker Fee Revenue

Apply the formula across all 25M exotic trades:
```
taker_fee = 0.07 × contracts × price × (1 - price)
```

Where `price` is in dollars (0 to 1), so we divide by 100 first.

In [ ]:
sql = """
SELECT
    COUNT(*)                                                              AS total_trades,
    SUM(contracts_traded)                                                 AS total_notional_usd,
    SUM(contracts_traded * price / 100.0)                                 AS total_handle_usd,
    SUM(0.07 * contracts_traded * (price / 100.0) * (1 - price / 100.0)) AS total_taker_fees_usd,
    100.0 * SUM(0.07 * contracts_traded * (price / 100.0) * (1 - price / 100.0)) 
          / NULLIF(SUM(contracts_traded), 0)                               AS fee_pct_of_notional,
    100.0 * SUM(0.07 * contracts_traded * (price / 100.0) * (1 - price / 100.0)) 
          / NULLIF(SUM(contracts_traded * price / 100.0), 0)               AS fee_pct_of_handle
FROM kalshi.trade_report
WHERE starts_with(report_ticker, 'KXMVE')
  AND contracts_traded > 0
  AND price > 0
"""

df_total, qid = run_fresh(sql, "Notebook 04 - Total Taker Fees")
print(f"Saved as Dune query {qid}")
df_total.to_string()